# 02 - Limpeza dos casos

**Objetivo:** deixar os dados do InfoDengue prontos para análise e modelagem.

Fonte: `data/raw/infodengue_teresina.csv` (baixado no `01_coleta`). O arquivo bruto **não é alterado**: tudo o que for tratado aqui será salvo em `data/processed/`.

## 1. Converter a data e ordenar no tempo

A API devolve um CSV com as semanas da mais recente pra mais antiga, e a data (`data_iniSE`) vem como texto. Converti e ordenei antes de qualquer coisa: com a ordem invertida, o lag de “semana anterior” pegaria a semana seguinte, e o modelo ia treinar vendo o futuro.



In [1]:
import pandas as pd

# Lê o dado bruto 
casos = pd.read_csv("../data/raw/infodengue_teresina.csv")
casos.shape

(615, 30)

In [2]:
# data_iniSE é texto e a primeira linha é a semana mais recente
print(casos["data_iniSE"].dtype)
casos[["data_iniSE", "SE", "casos", "casos_est"]].head(3)

str


,data_iniSE,SE,casos,casos_est
0,2026-10-11,202641,14,48.0
1,2026-10-04,202640,17,42.0
2,2026-09-27,202639,20,39.0


In [3]:
# Converte o texto "2015-01-04" para formato data
# format diz o padrão do texto: %Y = ano, %m = mês, %d = dia
# Com o format fixo, uma data fora do padrão dá erro em vez de virar data errada
casos["data_iniSE"] = pd.to_datetime(casos["data_iniSE"], format="%Y-%m-%d")
casos["data_iniSE"].dtype

dtype('<M8[us]')

In [4]:
# Ordena do mais antigo para o mais recente
casos = casos.sort_values("data_iniSE")

# Depois de ordenar, o índice continua com os números antigos (613, 612, ...)
# reset_index renumera de 0 em diante. drop=True joga fora o índice velho em vez de virar coluna
casos = casos.reset_index(drop=True)

casos[["data_iniSE", "SE", "casos", "casos_est"]].head(3)

,data_iniSE,SE,casos,casos_est
0,2015-01-04,201501,50,50.0
1,2015-01-11,201502,34,34.0
2,2015-01-18,201503,51,51.0


In [5]:
# Confere se as últimas linhas são as semanas mais recentes
casos[["data_iniSE", "SE", "casos", "casos_est"]].tail(3)

,data_iniSE,SE,casos,casos_est
612,2026-09-27,202639,20,39.0
613,2026-10-04,202640,17,42.0
614,2026-10-11,202641,14,48.0


In [6]:
# Checa se tá em ordem crescente, sem datas repetidas e sem perder linha
print("Ordem crescente:", casos["data_iniSE"].is_monotonic_increasing)
print("Datas repetidas:", casos["data_iniSE"].duplicated().sum())
print("Formato:", casos.shape)

Ordem crescente: True
Datas repetidas: 0
Formato: (615, 30)


### Conclusões do passo 1

- `data_iniSE` agora é do tipo data (`datetime64`).
- As linhas estão em ordem cronológica: de 04/01/2015 (linha 0) até 11/10/2026 (última linha).
- Nenhuma data repetida e o formato continua 615 × 30 (nenhuma linha perdida).

## 2. Escolher as colunas

Das 30 colunas, fiquei só com o essencial para o MVP:

| coluna | por que fica |
|---|---|
| `data_iniSE` | data de início da semana (eixo do tempo) |
| `SE` | semana epidemiológica no formato AAAASS |
| `casos_est` | **alvo** do modelo (corrige o atraso de notificação) |
| `casos` | casos notificados; serve para analisar o atraso (`casos_est − casos`) |

Tirei as colunas vazias ou com um valor só, os indicadores que o InfoDengue calcula a partir dos casos (Rt, nivel etc., que podem vazar informação do futuro) e o clima, que vou pegar do INMET pro período inteiro.

O arquivo bruto não muda: se alguma coluna fizer falta depois, ela continua em `data/raw/`.

In [7]:
# Lista das colunas que ficam
# Listar o que fica, em vez do que sai, deixa claro o que o resto do projeto usa
colunas_mantidas = ["data_iniSE", "SE", "casos", "casos_est"]
casos = casos[colunas_mantidas]
casos.head(3)

,data_iniSE,SE,casos,casos_est
0,2015-01-04,201501,50,50.0
1,2015-01-11,201502,34,34.0
2,2015-01-18,201503,51,51.0


In [8]:
# Verifica 4 colunas, mesmas linhas de antes e nada faltando
print("Formato:", casos.shape)
print(casos.dtypes)
print(casos.isna().sum())
casos.tail(3)

Formato: (615, 4)
data_iniSE    datetime64[us]
SE                     int64
casos                  int64
casos_est            float64
dtype: object
data_iniSE    0
SE            0
casos         0
casos_est     0
dtype: int64


,data_iniSE,SE,casos,casos_est
612,2026-09-27,202639,20,39.0
613,2026-10-04,202640,17,42.0
614,2026-10-11,202641,14,48.0


### Conclusões do passo 2

- Fiquei com 4 colunas: `data_iniSE`, `SE`, `casos` e `casos_est`.
- Nenhuma linha foi perdida e nenhuma das 4 colunas tem valor faltando.
- O dado bruto continua com as 30 colunas.

## 3. Marcar as semanas abertas

As semanas mais recentes recebem notificações atrasadas. O InfoDengue só corrige as 10 últimas (onde `casos_est` é diferente de `casos`), mas comparando o download com dados até 04/10/2026 com o download com dados até 11/10/2026, as 20 últimas semanas mudaram, inclusive semanas em que `casos_est` já era igual a `casos`. Marquei essas 20 como abertas: ficam de fora do treino e avaliação, mas continuam como entrada na previsão.

É uma comparação só, então esse N pode ser revisado se eu baixar os dados de novo.

In [9]:
# Quantas semanas mais recentes ainda estão "abertas" (os números ainda podem mudar)
# Valor tirado da comparação entre dois downloads
N_SEMANAS_ABERTAS = 20

# A tabela tá em ordem cronológica, então as N últimas linhas são as N semanas mais recentes
data_limite = casos["data_iniSE"].iloc[-N_SEMANAS_ABERTAS]

# Coluna True/False: aberta se a semana começa na data limite ou depois
# Não apaga as linhas: ficam fora do treino e da avaliação, mas servem de entrada na previsão real
casos["semana_aberta"] = casos["data_iniSE"] >= data_limite

In [10]:
# Confere se são exatamente N semanas abertas, todas no fim da tabela
print("Primeira semana aberta:", data_limite.date())
print(casos["semana_aberta"].value_counts())

# Mostra as 2 últimas semanas fechadas e as abertas
casos.tail(N_SEMANAS_ABERTAS + 2)

Primeira semana aberta: 2026-05-31
semana_aberta
False    595
True      20
Name: count, dtype: int64


,data_iniSE,SE,casos,casos_est,semana_aberta
593,2026-05-17,202620,691,691.0,False
594,2026-05-24,202621,849,849.0,False
595,2026-05-31,202622,925,925.0,True
596,2026-06-07,202623,697,697.0,True
597,2026-06-14,202624,632,632.0,True
598,2026-06-21,202625,374,374.0,True
599,2026-06-28,202626,335,335.0,True
600,2026-07-05,202627,285,285.0,True
601,2026-07-12,202628,181,181.0,True
602,2026-07-19,202629,172,172.0,True


### Conclusões do passo 3

- Criei a coluna `semana_aberta`: 20 semanas abertas (a partir de 31/05/2026) e 595 fechadas.
- As abertas continuam no dado, mas ficam fora do treino e da avaliação.

## 4. Salvar o dado tratado

Salvo em `data/processed/casos_teresina.csv`, sempre gerado por este notebook. As próximas etapas leem daqui. Como CSV não guarda tipo, a leitura precisa de `parse_dates=["data_iniSE"]`.


In [11]:
# Salva o dado tratado
CAMINHO_PROCESSED = "../data/processed/casos_teresina.csv"

# index=False pra não salvar o índice (0, 1, 2...) como uma coluna a mais
casos.to_csv(CAMINHO_PROCESSED, index=False)

In [12]:
# Confere lendo o arquivo de volta
# CSV é texto puro: sem parse_dates, a data_iniSE voltaria como str
relido = pd.read_csv(CAMINHO_PROCESSED, parse_dates=["data_iniSE"])
print(relido.dtypes)

# True se o arquivo tem exatamente o mesmo conteúdo e tipos que o DataFrame da memória
print("Igual ao da memória:", relido.equals(casos))
relido.shape

data_iniSE       datetime64[us]
SE                        int64
casos                     int64
casos_est               float64
semana_aberta              bool
dtype: object
Igual ao da memória: True


(615, 5)

### Conclusões do passo 4

- Dado limpo salvo em `data/processed/casos_teresina.csv`: 615 semanas, 5 colunas.
- Relido do disco e conferido: idêntico ao da memória.
- A próxima etapa (clima do INMET) parte desse arquivo.